# Debugger de pipeline SIREN

Ce notebook permet de :
- charger un petit fragment de données (parquet local ou extrait à partir du flux JOCAS si l’environnement est prêt) ;
- construire la pipeline de résolution SIREN ;
- inspecter les requêtes, les résultats et le cache ;
- déboguer facilement une entreprise ou un cas problématique.

In [ ]:
from pathlib import Path
import os
import sys

import pandas as pd

ROOT = Path.cwd().resolve()
if (ROOT / 'src').exists():
    sys.path.insert(0, str(ROOT / 'src'))

from siren_resolver import (
    Address,
    CompanyQuery,
    CONTRACTING_COLUMNS,
    ParquetSirenCache,
    ResolverConfig,
    SirenResolver,
    GoogleCSEProvider,
    RechercheEntreprisesProvider,
    SirenResolutionPipeline,
)

print('Workspace:', ROOT)
print('Python path ready.')

In [ ]:
# Configuration de base
DATA_DIR = Path(os.getenv('SIREN_DATA_DIR', ROOT / 'tmp_siren_debug'))
DATA_DIR.mkdir(parents=True, exist_ok=True)

sample_df = None

# 1) Essayer de charger un parquet local si disponible
candidate_files = list(ROOT.glob('**/*.parquet'))
if candidate_files:
    sample_path = candidate_files[0]
    print(f'Chargement du parquet local : {sample_path}')
    sample_df = pd.read_parquet(sample_path).head(20)

# 2) Sinon, essayer de télécharger un petit fragment depuis le bucket JOCAS si l'environnement est prêt
if sample_df is None:
    try:
        from jocas_common import connect_jocas
        print('Connexion au bucket JOCAS...')
        conn = connect_jocas()
        sample_df = conn.execute(
            "SELECT entreprise_nom, location_label, location_zipcode, location_departement FROM jocas WHERE entreprise_nom IS NOT NULL LIMIT 20"
        ).df()
        sample_df = sample_df.rename(columns={
            'entreprise_nom': 'CONTRACTING_STATED_NAME',
            'location_label': 'CONTRACTING_ADDRESS',
            'location_zipcode': 'location_zipcode',
            'location_departement': 'location_departement',
        })
        print('Fragment JOCAS chargé avec succès.')
    except Exception as exc:
        print(f'Impossible de charger depuis JOCAS : {exc}')

# 3) Sinon, utiliser un petit dataframe de démonstration
if sample_df is None:
    print('Utilisation d'un échantillon synthétique.')
    sample_df = pd.DataFrame([
        {'CONTRACTING_STATED_NAME': 'ACME SA', 'CONTRACTING_ADDRESS': '12 rue des Lilas 75002 Paris', 'CONTRACTING_SIREN': None},
        {'CONTRACTING_STATED_NAME': 'Société Exemple', 'CONTRACTING_ADDRESS': '3 avenue de la République 69002 Lyon', 'CONTRACTING_SIREN': None},
        {'CONTRACTING_STATED_NAME': 'Mairie de Bordeaux', 'CONTRACTING_ADDRESS': '1 place de la Bourse 33000 Bordeaux', 'CONTRACTING_SIREN': None},
    ])

sample_df.head()

In [ ]:
# Builder de pipeline et objets de résolution
config = ResolverConfig()
config = ResolverConfig(
    pipeline=config.pipeline,
    recherche_entreprises=config.recherche_entreprises,
    google_cse=config.google_cse,
)

cache = ParquetSirenCache(
    path=DATA_DIR / 'debug_cache.parquet',
    name_col=CONTRACTING_COLUMNS.name_col,
    address_col=CONTRACTING_COLUMNS.address_col,
    siren_col=CONTRACTING_COLUMNS.siren_col,
)

providers = [
    RechercheEntreprisesProvider(config.recherche_entreprises, config.pipeline.min_match_score),
    GoogleCSEProvider(config.google_cse),
]
resolver = SirenResolver(cache=cache, providers=providers)
pipeline = SirenResolutionPipeline(config=config, resolver=resolver, cache=cache, columns=CONTRACTING_COLUMNS)

print('Cache initial:', len(cache))
print('Providers actifs:')
for provider in providers:
    print('-', provider.name, '| available =', provider.is_available)

In [ ]:
# Transformer les lignes en CompanyQuery
def to_query(row):
    row_dict = row._asdict()
    raw_name = row_dict[CONTRACTING_COLUMNS.name_col]
    address = CONTRACTING_COLUMNS.build_address(row_dict)
    return CompanyQuery(raw_name=str(raw_name), address=address, role=CONTRACTING_COLUMNS.role)

queries = [to_query(row) for row in sample_df.itertuples(index=False)]
for idx, q in enumerate(queries):
    print(idx, '|', q.raw_name, '|', q.address)

In [ ]:
# Résolution ligne par ligne pour debug
for q in queries:
    result = resolver.resolve(q)
    print('---')
    print(q.raw_name)
    print('address:', q.address)
    print('siren:', result.siren)
    print('confidence:', result.confidence.value)
    print('match_score:', result.match_score)
    print('matched_name:', result.matched_name)

In [ ]:
# Exécution de la pipeline complète sur l'échantillon
sample_path = DATA_DIR / 'sample_contracting_debug.parquet'
sample_df.to_parquet(sample_path, index=False)

resolved_path = DATA_DIR / 'resolved_contracting_debug.parquet'
resolved_df = pipeline.run(missing_input_path=sample_path, resolved_output_path=resolved_path)

print('Résultat sauvegardé dans :', resolved_path)
resolved_df.head()

## Astuces de debug

- Modifiez la liste `sample_df` avec un cas précis à tester.
- Si vous avez déjà un parquet local, le notebook le charge automatiquement.
- Pour un vrai jeu de données, remplacez `sample_df` par un sous-échantillon extrait de votre base, puis relancez la cellule de pipeline.
- Le cache est sérialisé dans le dossier temporaire défini par `SIREN_DATA_DIR` pour pouvoir l’inspecter entre plusieurs exécutions.